In [2]:
%pip install transformers torch pandas


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd

# ==========================================
# PHASE 1: DATA PREPARATION
# ==========================================

class ParaphraseDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=128):
        """
        Custom Dataset to handle the PAWS/MRPC CSV format.
        """
        # Load data using pandas
        self.data = pd.read_csv(csv_file)
        
        # Check for correct columns (based on your screenshots)
        required_cols = ['sentence1', 'sentence2', 'label']
        if not all(col in self.data.columns for col in required_cols):
            raise ValueError(f"CSV must contain columns: {required_cols}")
            
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Tokenize Sentence A
        sent_a = self.tokenizer(
            str(row['sentence1']), 
            padding='max_length', 
            truncation=True, 
            max_length=self.max_length, 
            return_tensors="pt"
        )
        
        # Tokenize Sentence B
        sent_b = self.tokenizer(
            str(row['sentence2']), 
            padding='max_length', 
            truncation=True, 
            max_length=self.max_length, 
            return_tensors="pt"
        )

        return {
            'input_ids_a': sent_a['input_ids'].squeeze(0),
            'attention_mask_a': sent_a['attention_mask'].squeeze(0),
            'input_ids_b': sent_b['input_ids'].squeeze(0),
            'attention_mask_b': sent_b['attention_mask'].squeeze(0),
            'label': torch.tensor(row['label'], dtype=torch.float)
        }

In [13]:
# ==========================================
# PHASE 2: SIAMESE MODEL ARCHITECTURE
# ==========================================

class SiameseNetwork(nn.Module):
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        super(SiameseNetwork, self).__init__()
        
        # Load the pre-trained Transformer (The "Shared Weights")
        self.transformer = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        """
        Converts the list of token vectors into a single sentence vector.
        It averages the vectors but ignores padding tokens.
        """
        token_embeddings = model_output.last_hidden_state # [Batch, Seq_Len, 768]
        
        # Expand mask to match embedding dimensions
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        
        # Sum embeddings and divide by the number of non-padding tokens
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        """
        Forward pass for a SINGLE sentence. 
        We call this twice during training (once for A, once for B).
        """
        output = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        embedding = self.mean_pooling(output, attention_mask)
        return embedding

In [8]:
df = pd.read_csv('paws_train.csv')
df

FileNotFoundError: [Errno 2] No such file or directory: 'paws_train.csv'

In [ ]:
# ==========================================
# DRIVER CODE (How to run it)
# ==========================================

# 1. Setup Tokenizer & Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# 2. Load Dataset (Replace with your actual file path)
# Assuming 'paws_train.csv' has columns: sentence1, sentence2, label
dataset = ParaphraseDataset('paws_train.csv', tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 3. Initialize Model
model = SiameseNetwork().to(device)

# 4. Test a single batch to verify shapes
sample_batch = next(iter(dataloader))

# Move inputs to GPU/CPU
ids_a = sample_batch['input_ids_a'].to(device)
mask_a = sample_batch['attention_mask_a'].to(device)
ids_b = sample_batch['input_ids_b'].to(device)
mask_b = sample_batch['attention_mask_b'].to(device)

# Get Embeddings for both sentences
embedding_a = model(ids_a, mask_a) # Shape: [16, 768]
embedding_b = model(ids_b, mask_b) # Shape: [16, 768]

print(f"Embedding Shape: {embedding_a.shape}") # Should be [Batch_Size, 768]
print("Phase 1 & 2 Setup Complete.")

FileNotFoundError: [Errno 2] No such file or directory: 'paws_train.csv'

In [ ]:
# ==========================================
# PHASE 3: TRAINING LOOP
# ==========================================

import torch.optim as optim

def train_model(model, dataloader, epochs=3):
    # 1. Define Loss Function
    # margin=0.5 means: "If they are different, push them apart until 
    # their distance is at least 0.5".
    criterion = nn.CosineEmbeddingLoss(margin=0.5)
    
    # 2. Optimizer (AdamW is standard for Transformers)
    optimizer = optim.AdamW(model.parameters(), lr=2e-5)

    model.train() # Set model to training mode
    
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(epochs):
        total_loss = 0
        
        for batch_idx, batch in enumerate(dataloader):
            # Move data to GPU/CPU
            ids_a = batch['input_ids_a'].to(device)
            mask_a = batch['attention_mask_a'].to(device)
            ids_b = batch['input_ids_b'].to(device)
            mask_b = batch['attention_mask_b'].to(device)
            labels = batch['label'].to(device)
            
            # -------------------------------------------
            # CRITICAL STEP: Adjust Labels for PyTorch
            # PyTorch's CosineEmbeddingLoss expects:
            #  1 = Similar
            # -1 = Dissimilar
            # Your CSV has 0 for dissimilar, so we convert 0 -> -1
            # -------------------------------------------
            targets = labels.clone()
            targets[targets == 0] = -1
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward Pass (Generate vectors)
            embedding_a = model(ids_a, mask_a)
            embedding_b = model(ids_b, mask_b)
            
            # Calculate Loss
            # "Push similar vectors close, push dissimilar vectors apart"
            loss = criterion(embedding_a, embedding_b, targets)
            
            # Backward Pass (Update weights)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(dataloader)
        print(f"--- Epoch {epoch+1} Complete. Average Loss: {avg_loss:.4f} ---")

    # Save the model
    torch.save(model.state_dict(), 'plagiarism_detector_model.pth')
    print("Model saved successfully.")

# ==========================================
# DRIVER CODE UPDATE
# ==========================================

# (Assuming 'model' and 'dataloader' are already defined from Phase 1 & 2)

# Run the training
# train_model(model, dataloader, epochs=3)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# ==========================================
# PHASE 4: INFERENCE & CHUNKING
# ==========================================

class PlagiarismChecker:
    def __init__(self, model, tokenizer, window_size=256, stride=200):
        """
        window_size: How many words per chunk (BERT limit is ~512 tokens).
        stride: How much to move the window (overlap helps catch sentences split in half).
        """
        self.model = model
        self.tokenizer = tokenizer
        self.window_size = window_size
        self.stride = stride
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.eval() # Set to evaluation mode (turns off Dropout)

    def chunk_text(self, text):
        """
        Splits a long string into overlapping chunks.
        """
        words = text.split()
        chunks = []
        
        # Sliding window logic
        for i in range(0, len(words), self.stride):
            chunk = " ".join(words[i : i + self.window_size])
            chunks.append(chunk)
            
            # Stop if we've reached the end
            if i + self.window_size >= len(words):
                break
        return chunks

    def get_document_embedding(self, text):
        """
        Converts a document into a LIST of vector embeddings (one for each chunk).
        """
        chunks = self.chunk_text(text)
        chunk_embeddings = []

        with torch.no_grad(): # No gradients needed for inference
            for chunk in chunks:
                # Tokenize
                inputs = self.tokenizer(
                    chunk, 
                    padding='max_length', 
                    truncation=True, 
                    max_length=512, 
                    return_tensors="pt"
                )
                
                # Move to GPU
                input_ids = inputs['input_ids'].to(self.device)
                mask = inputs['attention_mask'].to(self.device)

                # Forward pass (reuse the Siamese Network's forward method)
                vector = self.model(input_ids, mask) 
                
                # Normalize vector (Critical for Cosine Similarity!)
                vector = F.normalize(vector, p=2, dim=1)
                chunk_embeddings.append(vector.cpu())

        # Stack into a single tensor: Shape [Num_Chunks, 768]
        if chunk_embeddings:
            return torch.vstack(chunk_embeddings)
        else:
            return torch.zeros(1, 768)

    def compare_documents(self, doc_a_vectors, doc_b_vectors):
        """
        Compares two students.
        Returns the score of the MOST similar chunks (Max Similarity).
        """
        # doc_a: [Chunks_A, 768]
        # doc_b: [Chunks_B, 768]
        
        # Matrix Multiplication creates a similarity matrix of all chunks
        # sim_matrix[i][j] = similarity between Chunk i of A and Chunk j of B
        sim_matrix = torch.mm(doc_a_vectors, doc_b_vectors.T) # Shape: [Chunks_A, Chunks_B]
        
        # Find the single highest match in the entire document pair
        max_similarity = torch.max(sim_matrix).item()
        
        return max_similarity

# ==========================================
# DRIVER CODE (How to use it on real files)
# ==========================================

# 1. Dummy Student Data
student_submissions = {
    "Student_A": "Artificial intelligence is intelligence demonstrated by machines, as opposed to natural intelligence displayed by animals including humans.",
    "Student_B": "AI is defined as intelligence shown by machines, unlike natural intelligence which is seen in animals and humans.", # Paraphrased
    "Student_C": "The French Revolution was a period of radical political and societal change in France that began with the Estates General of 1789." # Unrelated
}

# 2. Load Model (If running separately from training)
# model.load_state_dict(torch.load('plagiarism_detector_model.pth'))
# model.to(device)

# 3. Initialize Checker
checker = PlagiarismChecker(model, tokenizer)

# 4. Pre-compute Embeddings for everyone
print("Computing embeddings...")
embeddings_db = {}
for name, text in student_submissions.items():
    embeddings_db[name] = checker.get_document_embedding(text)

# 5. Run All-vs-All Comparison
print("\nSimilarity Report:")
names = list(student_submissions.keys())

results = []

for i in range(len(names)):
    for j in range(i + 1, len(names)): # Compare every unique pair
        name_1 = names[i]
        name_2 = names[j]
        
        score = checker.compare_documents(embeddings_db[name_1], embeddings_db[name_2])
        
        print(f"{name_1} vs {name_2}: {score:.4f}")
        
        results.append((name_1, name_2, score))